# ✊ ✋ ✌️ Challenge 2: Rock Paper Scissors AI

Your job is to build an AI that plays Rock Paper Scissors.

The opponent chooses one move:

1. Rock
2. Paper
3. Scissors

The robot must choose the move that wins.

Rules:

- Paper beats Rock
- Scissors beats Paper
- Rock beats Scissors


In [ ]:
import numpy as np

input_names = ["Opponent Rock", "Opponent Paper", "Opponent Scissors"]
output_names = ["Robot Rock", "Robot Paper", "Robot Scissors"]

def show_move(features):
    emojis = ["✊ Rock", "✋ Paper", "✌️ Scissors"]
    for value, emoji in zip(features, emojis):
        if value == 1:
            return emoji
    return "Unknown"

def run_robot(features, weights, thresholds, show=True):
    features = np.array(features)
    output_values = features @ weights

    active_outputs = []
    for i, value in enumerate(output_values):
        if value >= thresholds[i]:
            active_outputs.append(i)

    if show:
        print("Opponent plays:", show_move(features))
        print("\nOutput values:")
        for name, value, threshold in zip(output_names, output_values, thresholds):
            print(f"{name}: {value:.2f}  (threshold: {threshold})")

        print("\nRobot prediction:")
        if len(active_outputs) == 1:
            print(output_names[active_outputs[0]])
        elif len(active_outputs) == 0:
            print("No move chosen.")
        else:
            print("Confused. Multiple moves:", [output_names[i] for i in active_outputs])

    return active_outputs, output_values

def winner(opponent_index, robot_index):
    # 0 = Rock, 1 = Paper, 2 = Scissors
    if opponent_index == robot_index:
        return "Draw"
    winning_robot_moves = {
        0: 1,  # opponent rock -> robot paper
        1: 2,  # opponent paper -> robot scissors
        2: 0,  # opponent scissors -> robot rock
    }
    if robot_index == winning_robot_moves[opponent_index]:
        return "Robot wins"
    return "Robot loses"


## Part 1 — Start with a broken robot

The robot below is not trained properly.

Run it and see what happens.


In [ ]:
# Opponent move:
# [Rock, Paper, Scissors]
opponent = [1, 0, 0]

weights = np.array([
    [1, 0, 0],   # Opponent Rock     -> [Robot Rock, Robot Paper, Robot Scissors]
    [0, 1, 0],   # Opponent Paper    -> [Robot Rock, Robot Paper, Robot Scissors]
    [0, 0, 1],   # Opponent Scissors -> [Robot Rock, Robot Paper, Robot Scissors]
])

thresholds = np.array([0.5, 0.5, 0.5])

run_robot(opponent, weights, thresholds)


## Part 2 — Train the robot manually

Edit the weight matrix so the robot always wins.

Hint:

- If opponent plays Rock, robot should play Paper.
- If opponent plays Paper, robot should play Scissors.
- If opponent plays Scissors, robot should play Rock.


In [ ]:
# EDIT THESE NUMBERS

weights = np.array([
    [1, 0, 0],   # Opponent Rock     -> [Robot Rock, Robot Paper, Robot Scissors]
    [0, 1, 0],   # Opponent Paper    -> [Robot Rock, Robot Paper, Robot Scissors]
    [0, 0, 1],   # Opponent Scissors -> [Robot Rock, Robot Paper, Robot Scissors]
])

thresholds = np.array([0.5, 0.5, 0.5])

def test_robot(weights, thresholds):
    tests = [
        ([1,0,0], 0),
        ([0,1,0], 1),
        ([0,0,1], 2),
    ]

    score = 0
    for features, opponent_index in tests:
        active_outputs, output_values = run_robot(features, weights, thresholds, show=False)

        if len(active_outputs) == 1:
            robot_index = active_outputs[0]
            result = winner(opponent_index, robot_index)
            if result == "Robot wins":
                score += 1
            print(f"Opponent: {input_names[opponent_index]:18s} | Robot: {output_names[robot_index]:15s} | {result}")
        elif len(active_outputs) == 0:
            print(f"Opponent: {input_names[opponent_index]:18s} | Robot: No move chosen    | Robot loses")
        else:
            print(f"Opponent: {input_names[opponent_index]:18s} | Robot: Confused          | Robot loses")

    print(f"\nScore: {score}/3")

test_robot(weights, thresholds)


## Part 3 — Hard mode: opponent probabilities

Now the input is not just one move. It is the robot's guess about what the opponent might play.

Example:

```python
opponent = [0.7, 0.2, 0.1]
```

means:

- 70% chance the opponent plays Rock
- 20% chance Paper
- 10% chance Scissors

Can your robot choose the best move?


In [ ]:
opponent_probability = [0.7, 0.2, 0.1]

active_outputs, output_values = run_robot(opponent_probability, weights, thresholds)

print("\nRaw output values:", output_values)
print("Best move by highest score:", output_names[np.argmax(output_values)])


## Part 4 — Build a cheating robot

Create a robot that wins against these predicted opponent patterns:

```text
Mostly Rock
Mostly Paper
Mostly Scissors
Mixed Rock/Paper
Mixed Paper/Scissors
```

Think carefully: should the robot choose one move using thresholds, or should it pick the highest output value?


In [ ]:
probability_tests = [
    ([0.8, 0.1, 0.1], "Mostly Rock"),
    ([0.1, 0.8, 0.1], "Mostly Paper"),
    ([0.1, 0.1, 0.8], "Mostly Scissors"),
    ([0.45, 0.45, 0.10], "Mixed Rock/Paper"),
    ([0.10, 0.45, 0.45], "Mixed Paper/Scissors"),
]

for features, description in probability_tests:
    output_values = np.array(features) @ weights
    best_move = np.argmax(output_values)
    print(f"{description:22s} -> Robot chooses {output_names[best_move]} | scores {output_values}")
